In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))
from plotting import (combine_outputs, pca_plots, plot_experiment_comparisons,
                       plot_orientation_kde)

# ── Experiment-specific config ────────────────────────────────────────────────
CONDITION_COLORS = {
    "static": "red", "flow": "dodgerblue",
    "high_flow": "mediumblue", "low_flow": "lightblue",
    "resevoir": "darkorange",
}

def find_condition(row, experiment_setup="old"):
    name = str(row["image_name"]).lower()
    if "static" in name:
        return "static"
    elif "resevoir" in name or "reservoir" in name:
        return "resevoir"
    elif experiment_setup == "old":
        return "flow" if "flow" in name else "NA"
    else:
        if any(f"valve {n}" in name for n in [1, 2, 7, 8]):
            return "high_flow"
        elif any(f"valve {n}" in name for n in [3, 4, 5, 6]):
            return "low_flow"
        return "NA"

In [ ]:
root_dir = Path(r"Z:\Bel\Maria\New_Run_Outputs")
output_dir = root_dir.parent / f"{root_dir.name}_output_data"
output_dir.mkdir(parents=True, exist_ok=True)

combined_analysis_metrics, combined_branch_metrics = combine_outputs(root_dir)

Found extra uncategorised csv Z:\Bel\Maria\New_Run_Outputs\batch_timings.csv
Found extra uncategorised csv Z:\Bel\Maria\New_Run_Outputs\old_hough\batch_timings.csv
Total files combined: 22 analysis files and 84216 branch files


In [ ]:
COLS_TO_DROP = [
    "median_sprout_and_branch_median_cs_area_um2",
    "median_internal_pore_area_um2",
    "median_internal_pore_max_inscribed_radius_um",
    "p90_minus_p10_internal_pore_area_um2",
    "total_internal_pore_count",
    "p90_minus_p10_internal_pore_max_inscribed_radius_um",
    "internal_pore_area_fraction_in_filled_vascular_area",
]
combined_analysis_metrics_minimal = combined_analysis_metrics.drop(columns=COLS_TO_DROP)

In [ ]:
significant_params = pca_plots(combined_analysis_metrics_minimal, CONDITION_COLORS, save_dir=output_dir)

In [ ]:
xorder = [c for c in CONDITION_COLORS if c in combined_analysis_metrics_minimal["experiment"].unique()]
plot_experiment_comparisons(combined_analysis_metrics_minimal, significant_params[:5],
                            xorder, "significant_changes", output_dir, CONDITION_COLORS)
plot_orientation_kde(combined_branch_metrics, CONDITION_COLORS, save_dir=output_dir)